# Eksperimen Multimodal Fusion FENOMIND

**Status: NOT TRAINED / NOT VALIDATED.**

Notebook ini siap dijalankan begitu tersedia dataset **berpasangan**: rekaman wajah (CSV OpenFace) dan teks jawaban dari **partisipan dan sesi yang sama**, dengan label depresi (mis. PHQ-9 ≥ 10).

Dataset yang ada sekarang tidak memenuhi syarat:
- LMVD (visual) tidak punya transkrip.
- DatasetIndikasiDepresi (teks) tidak punya video.

Memasangkan keduanya secara acak/artifisial **tidak valid** dan sengaja ditolak oleh `validate_pairs`.

## Format manifest
CSV dengan kolom: `participant_id, session_id, label, openface_csv, text`
(`openface_csv` atau `text` boleh kosong untuk sesi yang kehilangan satu modality).

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

MANIFEST = ROOT / 'data' / 'paired' / 'manifest.csv'   # <- ganti dengan lokasi dataset berpasangan
if not MANIFEST.exists():
    raise SystemExit(f'Dataset berpasangan belum ada di {MANIFEST}. Fusion tetap NOT TRAINED.')

In [ ]:
import numpy as np, pandas as pd
from fenomind.fusion.experiment import load_manifest, run_fusion_cv
from fenomind.visual.features import read_openface_csv, session_features
from fenomind.text.preprocess import clean_text
from fenomind.text.embeddings import encode

m = load_manifest(MANIFEST)
vis_meta = json.loads((ROOT / 'artifacts/visual/model_meta.json').read_text())
feats = vis_meta['features']

def visual_row(path):
    if not isinstance(path, str) or not path:
        return np.full(len(feats), np.nan)
    f = session_features(read_openface_csv(path))
    if f.get('q_too_few_valid_frames', 1) or f['q_valid_ratio'] < 0.5:
        return np.full(len(feats), np.nan)
    return np.array([f.get(k, np.nan) for k in feats], float)

Xv = np.vstack([visual_row(p) for p in m['openface_csv'].fillna('')])
texts = np.array([clean_text(t) if isinstance(t, str) else '' for t in m['text']], dtype=object)
E = encode([t or '.' for t in texts], 'minilm_multi', cache=False)
E[texts == ''] = np.nan
y = m['label'].astype(int).to_numpy()
groups = m['participant_id'].astype(str).to_numpy()
print(len(m), 'sesi;', int((~np.isnan(Xv).all(1)).sum()), 'punya visual;', int((texts != '').sum()), 'punya teks')

In [ ]:
import joblib
from sklearn.base import clone
visual_model = joblib.load(ROOT / 'artifacts/visual/model.joblib')   # arsitektur + hiperparameter; dilatih ulang per fold
text_model = joblib.load(ROOT / 'artifacts/text/model.joblib')
res = run_fusion_cv(Xv, texts, E, y, groups, clone(visual_model), clone(text_model), n_splits=5, seed=42)
for k, v in res.items():
    if k != 'paired_auc_gain_vs_best_unimodal':
        print(f"{k:15s} AUC={v['roc_auc']:.3f} BA={v['balanced_accuracy']:.3f} sens={v['recall_sensitivity']:.3f} spec={v['specificity']:.3f} cov={v['coverage']:.2f}")
print(json.dumps(res['paired_auc_gain_vs_best_unimodal'], indent=2))

## Interpretasi
- Fusion hanya dianggap bermanfaat bila CI 95% `auc_gain` berada di atas 0.
- Bila tidak, laporkan bahwa fusion **tidak** memberi tambahan dibanding unimodal terbaik.
- Setelah eksperimen, perbarui `artifacts/fusion/status.json` menjadi `TRAINED_NOT_VALIDATED`, dan `VALIDATED` hanya setelah uji pada data independen.